# Solución Big Data – ModaEstilo
Datos: `modaestilo_hechos.csv` (ventas y alquileres) y `modaestilo_devoluciones.csv`, exportados del OLAP **ModaEstilo_DW**.

## Parte 1. Esquema y volumen

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.moda")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.moda.datos")
ruta = "/Volumes/workspace/moda/datos"
display(dbutils.fs.ls(ruta))     # aquí deben estar los 2 CSV

path,name,size,modificationTime
dbfs:/Volumes/workspace/moda/datos/exp_esquema/,exp_esquema/,0,1795830417932
dbfs:/Volumes/workspace/moda/datos/modaestilo_devoluciones.csv,modaestilo_devoluciones.csv,11032,1795814067000
dbfs:/Volumes/workspace/moda/datos/modaestilo_hechos.csv,modaestilo_hechos.csv,32369,1795814067000


## Parte 2. Lectura de los CSV en DataFrames de Spark (capa Bronze)

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType, DateType, DecimalType

# Esquema declarado (mejora M-07, resultado de EXP-12): importes en decimal exacto, fechas como date e ids como long
esquema_hechos = StructType([
    StructField("tipo_operacion", StringType(), True),
    StructField("id_operacion", LongType(), True),
    StructField("id_detalle", LongType(), True),
    StructField("nro_documento", StringType(), True),
    StructField("fuente", StringType(), True),
    StructField("fecha", DateType(), True),
    StructField("id_cliente", IntegerType(), True),
    StructField("distrito", StringType(), True),
    StructField("id_trabajador", IntegerType(), True),
    StructField("cargo", StringType(), True),
    StructField("id_articulo", IntegerType(), True),
    StructField("articulo", StringType(), True),
    StructField("categoria", StringType(), True),
    StructField("canal", StringType(), True),
    StructField("cantidad", IntegerType(), True),
    StructField("precio_unitario", DecimalType(12, 2), True),
    StructField("importe", DecimalType(12, 2), True),
    StructField("estado", StringType(), True),
])

esquema_devol = StructType([
    StructField("id_devolucion", IntegerType(), True),
    StructField("alquiler", StringType(), True),
    StructField("fecha_devolucion", DateType(), True),
    StructField("articulo", StringType(), True),
    StructField("categoria", StringType(), True),
    StructField("tipo_falla", StringType(), True),
    StructField("gravedad", StringType(), True),
    StructField("accion", StringType(), True),
    StructField("tipo_devolucion", StringType(), True),
    StructField("dias_retraso", IntegerType(), True),
    StructField("monto_cargo", DecimalType(12, 2), True),
])

dfHechos = (spark.read
  .format("csv")
  .option("header", "true")
  .schema(esquema_hechos)
  .load(f"{ruta}/modaestilo_hechos.csv"))

dfDevol = (spark.read
  .format("csv")
  .option("header", "true")
  .schema(esquema_devol)
  .load(f"{ruta}/modaestilo_devoluciones.csv"))

# overwriteSchema: reemplaza las tablas creadas antes con inferSchema (tenían importes double)
dfHechos.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.moda.bronze_hechos")
dfDevol.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.moda.bronze_devoluciones")

In [0]:
dfHechos.printSchema()
dfDevol.printSchema()

root
 |-- tipo_operacion: string (nullable = true)
 |-- id_operacion: long (nullable = true)
 |-- id_detalle: long (nullable = true)
 |-- nro_documento: string (nullable = true)
 |-- fuente: string (nullable = true)
 |-- fecha: date (nullable = true)
 |-- id_cliente: integer (nullable = true)
 |-- distrito: string (nullable = true)
 |-- id_trabajador: integer (nullable = true)
 |-- cargo: string (nullable = true)
 |-- id_articulo: integer (nullable = true)
 |-- articulo: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- canal: string (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- precio_unitario: decimal(12,2) (nullable = true)
 |-- importe: decimal(12,2) (nullable = true)
 |-- estado: string (nullable = true)

root
 |-- id_devolucion: integer (nullable = true)
 |-- alquiler: string (nullable = true)
 |-- fecha_devolucion: date (nullable = true)
 |-- articulo: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- tipo_falla: string

In [0]:
display(dfHechos)
print("Filas de hechos:", dfHechos.count(), "| Filas de devoluciones:", dfDevol.count())

tipo_operacion,id_operacion,id_detalle,nro_documento,fuente,fecha,id_cliente,distrito,id_trabajador,cargo,id_articulo,articulo,categoria,canal,cantidad,precio_unitario,importe,estado
VENTA,1,1,PV-0001,EXCEL,2026-01-03,6,San Borja,3,Vendedor,2,Pajarita satinada,Accesorios,WhatsApp,1,39.00,39.00,Vendido
VENTA,1,2,PV-0001,EXCEL,2026-01-03,6,San Borja,3,Vendedor,4,Pañuelo de bolsillo,Accesorios,WhatsApp,2,29.00,58.00,Vendido
VENTA,1,3,PV-0001,EXCEL,2026-01-03,6,San Borja,3,Vendedor,21,Vestido de gala satinado,Vestidos,WhatsApp,1,470.00,470.00,Vendido
VENTA,2,4,PV-0002,EXCEL,2026-01-09,31,La Molina,2,Vendedor,2,Pajarita satinada,Accesorios,WhatsApp,1,35.10,35.10,Vendido
VENTA,3,5,PV-0003,EXCEL,2026-01-15,18,Miraflores,3,Vendedor,10,Camisa Formal,Camisas,Web,1,90.00,90.00,Vendido
VENTA,4,6,PV-0004,EXCEL,2026-01-19,26,San Borja,2,Vendedor,3,Set de gemelos,Accesorios,WhatsApp,3,67.50,202.50,Vendido
VENTA,4,7,PV-0004,EXCEL,2026-01-19,26,San Borja,2,Vendedor,6,Saco cruzado,Blazers y sacos,WhatsApp,1,340.00,340.00,Vendido
VENTA,4,8,PV-0004,EXCEL,2026-01-19,26,San Borja,2,Vendedor,14,Vestido infantil de tul,Trajes infantiles,WhatsApp,1,171.00,171.00,Vendido
VENTA,5,9,PV-0005,EXCEL,2026-01-24,12,Los Olivos,3,Vendedor,13,Terno infantil,Trajes infantiles,WhatsApp,1,218.50,218.50,Vendido
VENTA,5,10,PV-0005,EXCEL,2026-01-24,12,Los Olivos,3,Vendedor,17,Terno tres piezas,Ternos,WhatsApp,1,620.00,620.00,Vendido


Filas de hechos: 250 | Filas de devoluciones: 107


## Parte 3. Análisis exploratorio (EDA)

In [0]:
from pyspark.sql import functions as F

# Estadísticas básicas
display(dfHechos.describe("cantidad", "precio_unitario", "importe"))

summary,cantidad,precio_unitario,importe
count,250,250,250
mean,1.096,203.540200,208.697600
stddev,0.3890881111431351,164.26791875007692,161.50948650517304
min,1,6.00,6.00
max,3,680.00,680.00


In [0]:
# Valores nulos por columna
display(dfHechos.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in dfHechos.columns]))

tipo_operacion,id_operacion,id_detalle,nro_documento,fuente,fecha,id_cliente,distrito,id_trabajador,cargo,id_articulo,articulo,categoria,canal,cantidad,precio_unitario,importe,estado
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
# Operaciones, unidades e importe por tipo
display(dfHechos.groupBy("tipo_operacion")
        .agg(F.countDistinct("id_operacion").alias("operaciones"),
             F.sum("cantidad").alias("unidades"),
             F.round(F.sum("importe"), 2).alias("importe")))

tipo_operacion,operaciones,unidades,importe
ALQUILER,100,125,14118.00
VENTA,79,149,38056.40


In [0]:
# Importe por categoría y por canal
display(dfHechos.groupBy("categoria").agg(F.round(F.sum("importe"), 2).alias("importe")).orderBy(F.desc("importe")))
display(dfHechos.groupBy("canal").count().orderBy(F.desc("count")))

categoria,importe
Ternos,20832.00
Vestidos,17433.00
Blazers y sacos,4506.75
Calzado,2751.25
Camisas,2441.50
Trajes infantiles,2427.50
Accesorios,1782.40


canal,count
WhatsApp,92
Instagram,65
Facebook,38
Web,38
Tienda física,17


In [0]:
# Valores distintos y posibles duplicados
display(dfHechos.select("estado").distinct())
print("Filas:", dfHechos.count(), "| Filas distintas:", dfHechos.distinct().count())

estado
Devuelto
Vencido
Vendido


Filas: 250 | Filas distintas: 250


In [0]:
# Muestra aproximada del 10 % (con semilla para repetir el resultado)
muestra = dfHechos.sample(0.10, seed=42)
print("Filas en la muestra:", muestra.count())
display(muestra)

Filas en la muestra: 24


tipo_operacion,id_operacion,id_detalle,nro_documento,fuente,fecha,id_cliente,distrito,id_trabajador,cargo,id_articulo,articulo,categoria,canal,cantidad,precio_unitario,importe,estado
VENTA,1,3,PV-0001,EXCEL,2026-01-03,6,San Borja,3,Vendedor,21,Vestido de gala satinado,Vestidos,WhatsApp,1,470.00,470.00,Vendido
VENTA,5,10,PV-0005,EXCEL,2026-01-24,12,Los Olivos,3,Vendedor,17,Terno tres piezas,Ternos,WhatsApp,1,620.00,620.00,Vendido
VENTA,22,34,PV-0022,EXCEL,2026-04-17,37,Breña,3,Vendedor,11,Camisa cuello italiano,Camisas,Instagram,1,120.00,120.00,Vendido
VENTA,28,44,PV-0028,EXCEL,2026-05-06,2,Comas,3,Vendedor,20,Vestido Elegante,Vestidos,Web,1,315.00,315.00,Vendido
VENTA,41,66,PV-0041,EXCEL,2026-06-25,5,San Isidro,2,Vendedor,24,Vestido de fiesta encaje,Vestidos,WhatsApp,1,455.00,455.00,Vendido
VENTA,52,82,PV-0052,EXCEL,2026-08-19,31,La Molina,2,Vendedor,13,Terno infantil,Trajes infantiles,Web,1,230.00,230.00,Vendido
VENTA,64,102,PV-0064,EXCEL,2026-10-13,15,Jesús María,2,Vendedor,16,Terno Ejecutivo,Ternos,WhatsApp,1,500.00,500.00,Vendido
VENTA,66,106,PV-0066,EXCEL,2026-10-20,16,San Isidro,3,Vendedor,12,Camisa Oxford,Camisas,Facebook,1,105.00,105.00,Vendido
VENTA,74,118,PV-0074,EXCEL,2026-11-17,39,Surco,2,Vendedor,15,Terno Clasico,Ternos,WhatsApp,1,450.00,450.00,Vendido
VENTA,75,119,PV-0082,EXCEL,2026-07-26,31,La Molina,2,Vendedor,10,Camisa Formal,Camisas,Facebook,1,120.00,120.00,Vendido


In [0]:
# Devoluciones: tipo de devolución y tipo de falla
display(dfDevol.groupBy("tipo_devolucion").count().orderBy(F.desc("count")))
display(dfDevol.filter(F.col("tipo_falla") != "Ninguna")
        .groupBy("tipo_falla", "gravedad").agg(F.count("*").alias("prendas"), F.sum("monto_cargo").alias("cargos")))

tipo_devolucion,count
A tiempo,49
Con retraso,29
Con falla,20
Con retraso y falla,9


tipo_falla,gravedad,prendas,cargos
Pérdida de accesorio,Grave,1,64.00
Botón faltante,Leve,5,53.00
Cierre dañado,Moderada,5,252.00
Rotura de tela,Grave,5,643.00
Daño no especificado,Leve,1,75.00
Costura abierta,Moderada,3,116.00
Mancha,Leve,9,251.00


## Parte 4. Limpieza y preparación (capa Silver)

In [0]:
dfHechos2 = (spark.table("workspace.moda.bronze_hechos")
  .withColumnRenamed("precio_unitario", "precio")
  .withColumnRenamed("importe", "monto")
  .select("tipo_operacion", "id_operacion", "id_detalle", "nro_documento", "fuente", "fecha",
          "distrito", "cargo", "articulo", "categoria", "canal", "cantidad", "precio", "monto", "estado")
  .withColumn("fecha", F.to_date("fecha"))
  .withColumn("monto", F.col("monto").cast("decimal(12,2)"))
  .withColumn("precio", F.col("precio").cast("decimal(12,2)"))
  .na.drop(subset=["id_detalle", "fecha", "monto"])
  .na.fill({"distrito": "No registrado", "canal": "No registrado"})
  .filter((F.col("cantidad") > 0) & (F.col("monto") > 0))
  .dropDuplicates(["tipo_operacion", "id_detalle"])
  .withColumn("anio", F.year("fecha"))
  .withColumn("mes", F.month("fecha"))
  .withColumn("anio_mes", F.date_format("fecha", "yyyy-MM")))

display(dfHechos2)

tipo_operacion,id_operacion,id_detalle,nro_documento,fuente,fecha,distrito,cargo,articulo,categoria,canal,cantidad,precio,monto,estado,anio,mes,anio_mes
VENTA,1,1,PV-0001,EXCEL,2026-01-03,San Borja,Vendedor,Pajarita satinada,Accesorios,WhatsApp,1,39.00,39.00,Vendido,2026,1,2026-01
VENTA,1,2,PV-0001,EXCEL,2026-01-03,San Borja,Vendedor,Pañuelo de bolsillo,Accesorios,WhatsApp,2,29.00,58.00,Vendido,2026,1,2026-01
VENTA,1,3,PV-0001,EXCEL,2026-01-03,San Borja,Vendedor,Vestido de gala satinado,Vestidos,WhatsApp,1,470.00,470.00,Vendido,2026,1,2026-01
VENTA,2,4,PV-0002,EXCEL,2026-01-09,La Molina,Vendedor,Pajarita satinada,Accesorios,WhatsApp,1,35.10,35.10,Vendido,2026,1,2026-01
VENTA,3,5,PV-0003,EXCEL,2026-01-15,Miraflores,Vendedor,Camisa Formal,Camisas,Web,1,90.00,90.00,Vendido,2026,1,2026-01
VENTA,4,6,PV-0004,EXCEL,2026-01-19,San Borja,Vendedor,Set de gemelos,Accesorios,WhatsApp,3,67.50,202.50,Vendido,2026,1,2026-01
VENTA,4,7,PV-0004,EXCEL,2026-01-19,San Borja,Vendedor,Saco cruzado,Blazers y sacos,WhatsApp,1,340.00,340.00,Vendido,2026,1,2026-01
VENTA,4,8,PV-0004,EXCEL,2026-01-19,San Borja,Vendedor,Vestido infantil de tul,Trajes infantiles,WhatsApp,1,171.00,171.00,Vendido,2026,1,2026-01
VENTA,5,9,PV-0005,EXCEL,2026-01-24,Los Olivos,Vendedor,Terno infantil,Trajes infantiles,WhatsApp,1,218.50,218.50,Vendido,2026,1,2026-01
VENTA,5,10,PV-0005,EXCEL,2026-01-24,Los Olivos,Vendedor,Terno tres piezas,Ternos,WhatsApp,1,620.00,620.00,Vendido,2026,1,2026-01


In [0]:
# Join: a cada alquiler se le agrega su devolución (tipo de devolución, falla y cargo)
dfDevol2 = (spark.table("workspace.moda.bronze_devoluciones")
  .withColumnRenamed("monto_cargo", "cargo_devolucion")
  .select("alquiler", "articulo", "fecha_devolucion", "tipo_devolucion", "tipo_falla", "gravedad", "cargo_devolucion"))

dfAlquileres = (dfHechos2.filter("tipo_operacion = 'ALQUILER'")
  .withColumn("alquiler", F.concat_ws("-", "fuente", "nro_documento"))
  .join(dfDevol2, on=["alquiler", "articulo"], how="left")
  .na.fill({"tipo_devolucion": "Sin devolver", "tipo_falla": "Sin devolver"}))

display(dfAlquileres.selectExpr("alquiler", "articulo", "estado", "tipo_devolucion", "tipo_falla", "cargo_devolucion"))

alquiler,articulo,estado,tipo_devolucion,tipo_falla,cargo_devolucion
EXCEL-RED-0003,Vestido largo plisado,Devuelto,A tiempo,Ninguna,0.00
EXCEL-RED-0035,Vestido de gala satinado,Devuelto,A tiempo,Ninguna,0.00
EXCEL-RED-0074,Vestido de cóctel,Devuelto,A tiempo,Ninguna,0.00
EXCEL-RED-0083,Blazer de lino,Vencido,Sin devolver,Sin devolver,null
EXCEL-RED-0085,Terno Clasico,Devuelto,A tiempo,Ninguna,0.00
EXCEL-RED-0010,Vestido de fiesta encaje,Devuelto,Con retraso,Ninguna,0.00
EXCEL-RED-0022,Saco cruzado,Devuelto,A tiempo,Ninguna,0.00
EXCEL-RED-0024,Terno tres piezas,Devuelto,A tiempo,Ninguna,0.00
EXCEL-RED-0031,Saco cruzado,Vencido,Sin devolver,Sin devolver,null
EXCEL-RED-0032,Vestido Elegante,Devuelto,A tiempo,Ninguna,0.00


In [0]:
# Comprobación: los totales deben cuadrar con ModaEstilo_DW (ventas S/ 38 056.40 y alquileres S/ 14 118.00)
totales = {r["tipo_operacion"]: float(r["total"])
           for r in dfHechos2.groupBy("tipo_operacion").agg(F.sum("monto").alias("total")).collect()}
print(totales)
assert abs(totales["VENTA"] - 38056.40) < 0.01 and abs(totales["ALQUILER"] - 14118.00) < 0.01
print("Silver cuadra con el OLAP")

dfHechos2.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.moda.silver_hechos")
dfAlquileres.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.moda.silver_alquileres")

{'VENTA': 38056.4, 'ALQUILER': 14118.0}
Silver cuadra con el OLAP


## Parte 5. Indicadores (capa Gold)

In [0]:
gold_mes = (dfHechos2.groupBy("anio_mes", "tipo_operacion")
  .agg(F.sum("monto").alias("ingreso"),
       F.countDistinct("id_operacion").alias("operaciones"),
       F.sum("cantidad").alias("unidades"))
  .withColumn("ticket_promedio", F.round(F.col("ingreso") / F.col("operaciones"), 2))
  .orderBy("anio_mes"))

gold_categoria = (dfHechos2.groupBy("categoria", "tipo_operacion")
  .agg(F.sum("monto").alias("ingreso"), F.sum("cantidad").alias("unidades")))

gold_canal = (dfHechos2.groupBy("canal")
  .agg(F.sum("monto").alias("ingreso"), F.countDistinct("id_operacion").alias("operaciones")))

gold_devoluciones = (dfAlquileres.groupBy("tipo_devolucion", "tipo_falla")
  .agg(F.count("*").alias("prendas"), F.sum("cargo_devolucion").alias("cargos")))

gold_mes.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.moda.gold_ingresos_mes")
gold_categoria.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.moda.gold_ingresos_categoria")
gold_canal.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.moda.gold_ingresos_canal")
gold_devoluciones.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.moda.gold_devoluciones")

## Parte 6. Visualizaciones (en cada resultado: + > Visualization)

In [0]:
%sql
-- Ingresos por mes (gráfico de líneas: X anio_mes, Y ingreso, agrupar por tipo_operacion)
SELECT anio_mes, tipo_operacion, ingreso, operaciones, ticket_promedio
FROM workspace.moda.gold_ingresos_mes
ORDER BY anio_mes

anio_mes,tipo_operacion,ingreso,operaciones,ticket_promedio
2026-01,VENTA,2784.10,6,464.02
2026-01,ALQUILER,817.00,7,116.71
2026-02,VENTA,2191.50,5,438.30
2026-02,ALQUILER,1028.00,7,146.86
2026-03,VENTA,2446.50,6,407.75
2026-03,ALQUILER,1238.00,8,154.75
2026-04,ALQUILER,1249.00,9,138.78
2026-04,VENTA,3983.75,9,442.64
2026-05,ALQUILER,1444.00,11,131.27
2026-05,VENTA,6294.75,9,699.42


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Ingresos por categoría (barras)
SELECT categoria, tipo_operacion, ingreso, unidades
FROM workspace.moda.gold_ingresos_categoria
ORDER BY ingreso DESC

categoria,tipo_operacion,ingreso,unidades
Ternos,VENTA,14021.00,27
Vestidos,VENTA,12718.00,32
Ternos,ALQUILER,6811.00,38
Vestidos,ALQUILER,4715.00,34
Blazers y sacos,VENTA,3141.75,10
Calzado,VENTA,2751.25,12
Camisas,VENTA,2077.50,21
Trajes infantiles,VENTA,1687.50,8
Accesorios,VENTA,1659.40,39
Blazers y sacos,ALQUILER,1365.00,16


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Ingresos por canal (barras o anillo)
SELECT canal, ingreso, operaciones
FROM workspace.moda.gold_ingresos_canal
ORDER BY ingreso DESC

canal,ingreso,operaciones
WhatsApp,17634.05,56
Instagram,12002.80,51
Facebook,10356.00,28
Web,8377.55,29
Tienda física,3804.00,15


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Devoluciones por tipo y falla (barras apiladas)
SELECT tipo_devolucion, tipo_falla, prendas, cargos
FROM workspace.moda.gold_devoluciones
ORDER BY prendas DESC

tipo_devolucion,tipo_falla,prendas,cargos
A tiempo,Ninguna,49,90.00
Con retraso,Ninguna,29,0.00
Sin devolver,Sin devolver,18,null
Con falla,Mancha,6,173.00
Con falla,Rotura de tela,4,495.00
Con falla,Cierre dañado,3,157.00
Con retraso y falla,Mancha,3,78.00
Con falla,Costura abierta,3,116.00
Con retraso y falla,Botón faltante,3,31.00
Con retraso y falla,Cierre dañado,2,95.00


Databricks visualization. Run in Databricks to view.